In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import matplotlib.pyplot as plt

In [2]:
ROOT = Path("..")

PROCESSED_DATA = ROOT / "data" / "processed"

features = pd.read_csv(
    PROCESSED_DATA / "enso_features_multihorizon.csv",
    parse_dates=["Date"]
)

features.head()

,Date,nino34,nino3,nino4,iod,soi,nino34_lag1,nino34_lag2,nino34_lag3,nino3_lag1,...,nino3_roll3,nino4_roll3,iod_roll3,soi_roll3,future_nino34_lead1,future_nino34_lead2,future_nino34_lead3,future_nino34_lead4,future_nino34_lead5,future_nino34_lead6
0,1951-04-01,-0.23,-0.21,-0.42,-0.09,-0.3,-0.38,-1.04,-1.30,-0.33,...,-0.433333,-0.703333,0.190000,0.166667,-0.01,0.00,0.30,0.17,0.51,0.49
1,1951-05-01,-0.01,-0.18,0.26,0.16,-0.7,-0.23,-0.38,-1.04,-0.21,...,-0.240000,-0.246667,0.110000,-0.366667,0.00,0.30,0.17,0.51,0.49,0.55
2,1951-06-01,0.00,0.04,0.08,-0.12,0.2,-0.01,-0.23,-0.38,-0.18,...,-0.116667,-0.026667,-0.016667,-0.266667,0.30,0.17,0.51,0.49,0.55,0.31
3,1951-07-01,0.30,0.62,0.23,-0.17,-1.0,0.00,-0.01,-0.23,0.04,...,0.160000,0.190000,-0.043333,-0.500000,0.17,0.51,0.49,0.55,0.31,0.13
4,1951-08-01,0.17,0.41,-0.26,-0.19,-0.2,0.30,0.00,-0.01,0.62,...,0.356667,0.016667,-0.160000,-0.333333,0.51,0.49,0.55,0.31,0.13,-0.01


In [3]:
feature_cols = [

    "nino34",
    "nino3",
    "nino4",
    "iod",
    "soi",

    "nino34_lag1",
    "nino34_lag2",
    "nino34_lag3",

    "nino3_lag1",
    "nino3_lag2",
    "nino3_lag3",

    "nino4_lag1",
    "nino4_lag2",
    "nino4_lag3",

    "iod_lag1",
    "iod_lag2",
    "iod_lag3",

    "soi_lag1",
    "soi_lag2",
    "soi_lag3",

    "nino34_roll3",
    "nino3_roll3",
    "nino4_roll3",
    "iod_roll3",
    "soi_roll3"
]

targets = [

    "future_nino34_lead1",
    "future_nino34_lead2",
    "future_nino34_lead3",
    "future_nino34_lead4",
    "future_nino34_lead5",
    "future_nino34_lead6"

]

In [5]:

print("Persistence Baseline")


for lead in range(1, 7):

    target = f"future_nino34_lead{lead}"

    baseline = features["nino34"]

    actual = features[target]

    mae = mean_absolute_error(actual, baseline)

    rmse = np.sqrt(
        mean_squared_error(actual, baseline)
    )

    corr = np.corrcoef(
        actual,
        baseline
    )[0, 1]

    print(f"\nLead {lead}")

    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("Correlation:", round(corr, 4))

Persistence Baseline

Lead 1
MAE : 0.2087
RMSE: 0.2655
Correlation: 0.953

Lead 2
MAE : 0.3554
RMSE: 0.45
Correlation: 0.865

Lead 3
MAE : 0.468
RMSE: 0.5952
Correlation: 0.7639

Lead 4
MAE : 0.562
RMSE: 0.7218
Correlation: 0.6527

Lead 5
MAE : 0.6492
RMSE: 0.8365
Correlation: 0.5339

Lead 6
MAE : 0.724
RMSE: 0.9405
Correlation: 0.4117


In [9]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


def evaluate_model(model, X, y, model_name):

    print(f"\n{model_name}")

    tscv = TimeSeriesSplit(n_splits=5)

    mae_scores = []
    rmse_scores = []
    corr_scores = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)

        rmse = np.sqrt(
            mean_squared_error(y_test, y_pred)
        )

        corr = np.corrcoef(
            y_test,
            y_pred
        )[0, 1]

        mae_scores.append(mae)
        rmse_scores.append(rmse)
        corr_scores.append(corr)

        print(f"\nFold {fold}")
        print("MAE :", round(mae, 4))
        print("RMSE:", round(rmse, 4))
        print("Corr:", round(corr, 4))

    print("\nAverage")
    print("MAE :", round(np.mean(mae_scores), 4))
    print("RMSE:", round(np.mean(rmse_scores), 4))
    print("Corr:", round(np.mean(corr_scores), 4))

In [10]:
for target in targets:

    X = features[feature_cols]
    y = features[target]

    # Linear Regression
    linear_model = LinearRegression()

    evaluate_model(
        linear_model,
        X,
        y,
        "Linear Regression"
    )

    # Random Forest
    rf_model = RandomForestRegressor(
        n_estimators=200,
        random_state=42
    )

    evaluate_model(
        rf_model,
        X,
        y,
        "Random Forest"
    )


Linear Regression

Fold 1
MAE : 0.2103
RMSE: 0.2643
Corr: 0.9602

Fold 2
MAE : 0.2115
RMSE: 0.2662
Corr: 0.9451

Fold 3
MAE : 0.1517
RMSE: 0.1973
Corr: 0.9792

Fold 4
MAE : 0.1475
RMSE: 0.1881
Corr: 0.9678

Fold 5
MAE : 0.1448
RMSE: 0.179
Corr: 0.9801

Average
MAE : 0.1732
RMSE: 0.219
Corr: 0.9665

Random Forest

Fold 1
MAE : 0.2564
RMSE: 0.3274
Corr: 0.9394

Fold 2
MAE : 0.2477
RMSE: 0.3157
Corr: 0.9187

Fold 3
MAE : 0.1935
RMSE: 0.2587
Corr: 0.9651

Fold 4
MAE : 0.1768
RMSE: 0.2248
Corr: 0.9534

Fold 5
MAE : 0.17
RMSE: 0.2158
Corr: 0.9719

Average
MAE : 0.2089
RMSE: 0.2685
Corr: 0.9497

Linear Regression

Fold 1
MAE : 0.3484
RMSE: 0.4411
Corr: 0.8837

Fold 2
MAE : 0.3477
RMSE: 0.4451
Corr: 0.8433

Fold 3
MAE : 0.2687
RMSE: 0.3443
Corr: 0.9342

Fold 4
MAE : 0.2623
RMSE: 0.34
Corr: 0.8933

Fold 5
MAE : 0.2585
RMSE: 0.3114
Corr: 0.9394

Average
MAE : 0.2971
RMSE: 0.3764
Corr: 0.8988

Random Forest

Fold 1
MAE : 0.4235
RMSE: 0.5273
Corr: 0.8354

Fold 2
MAE : 0.4121
RMSE: 0.5203
Corr: 0.

In [11]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

target = "future_nino34_lead1"

y = features[target]

exog = features[
    [
        "nino3",
        "nino4",
        "iod",
        "soi"
    ]
]
tscv = TimeSeriesSplit(n_splits=5)

mae_scores = []
rmse_scores = []
corr_scores = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(features), start=1):

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    exog_train = exog.iloc[train_idx]
    exog_test = exog.iloc[test_idx]

    model = SARIMAX(

        endog=y_train,

        exog=exog_train,

        order=(2,0,2),

        enforce_stationarity=False,

        enforce_invertibility=False

    )

    fitted = model.fit(disp=False)

    predictions = fitted.predict(

        start=y_test.index[0],

        end=y_test.index[-1],

        exog=exog_test

    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    corr = np.corrcoef(
        y_test,
        predictions
    )[0,1]

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    corr_scores.append(corr)

    print(f"\nFold {fold}")
    print("MAE :", round(mae,4))
    print("RMSE:", round(rmse,4))
    print("Corr:", round(corr,4))

print("\nAverage")

print("MAE :", np.mean(mae_scores))
print("RMSE:", np.mean(rmse_scores))
print("Corr:", np.mean(corr_scores))

/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Fold 1
MAE : 0.9902
RMSE: 1.1489
Corr: -0.1243


/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/krishnasahithidharani/miniforge3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Fold 2
MAE : 0.2777
RMSE: 0.3637
Corr: 0.9157

Fold 3
MAE : 0.2649
RMSE: 0.3502
Corr: 0.9564

Fold 4
MAE : 0.2323
RMSE: 0.2966
Corr: 0.9387

Fold 5
MAE : 0.2389
RMSE: 0.3128
Corr: 0.9671

Average
MAE : 0.4008085201980499
RMSE: 0.4944534353236172
Corr: 0.7307244830564571


In [12]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np

# -------------------------------------------------------
# Choose target
# -------------------------------------------------------

target = "future_nino34_lead1"

y = features[target]

exog = features[
    [
        "nino3",
        "nino4",
        "iod",
        "soi"
    ]
]

# -------------------------------------------------------
# ADF Test
# -------------------------------------------------------

adf_result = adfuller(y.dropna())

print("ADF Statistic :", adf_result[0])
print("p-value       :", adf_result[1])

if adf_result[1] < 0.05:

    d = 0

else:

    d = 1

print("Using d =", d)

# -------------------------------------------------------
# Candidate Orders
# -------------------------------------------------------

orders = [

    (1, d, 0),
    (1, d, 1),
    (2, d, 1),
    (2, d, 2),
    (3, d, 1),
    (3, d, 2)

]

# -------------------------------------------------------
# Time Series CV
# -------------------------------------------------------

tscv = TimeSeriesSplit(n_splits=5)

mae_scores = []
rmse_scores = []
corr_scores = []

for fold, (train_idx, test_idx) in enumerate(
    tscv.split(features),
    start=1
):

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    exog_train = exog.iloc[train_idx]
    exog_test = exog.iloc[test_idx]

    best_model = None
    best_order = None
    best_aic = np.inf

    # ---------------------------------------------
    # Search best ARIMAX order
    # ---------------------------------------------

    for order in orders:

        try:

            model = SARIMAX(

                endog=y_train,

                exog=exog_train,

                order=order,

                enforce_stationarity=False,

                enforce_invertibility=False

            )

            fitted = model.fit(

                disp=False,

                maxiter=300

            )

            if fitted.aic < best_aic:

                best_aic = fitted.aic

                best_model = fitted

                best_order = order

        except:

            continue

    print()

    print("Fold", fold)

    print("Best Order :", best_order)

    print("Best AIC   :", round(best_aic,2))

    predictions = best_model.predict(

        start=y_test.index[0],

        end=y_test.index[-1],

        exog=exog_test

    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    corr = np.corrcoef(
        y_test,
        predictions
    )[0,1]

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    corr_scores.append(corr)

    print("MAE :", round(mae,4))
    print("RMSE:", round(rmse,4))
    print("Corr:", round(corr,4))

print()

print("====================================")

print("Average")

print("====================================")

print("MAE :", round(np.mean(mae_scores),4))

print("RMSE:", round(np.mean(rmse_scores),4))

print("Corr:", round(np.mean(corr_scores),4))

ADF Statistic : -7.272544180337708
p-value       : 1.5746274663021938e-10
Using d = 0

Fold 1
Best Order : (1, 0, 0)
Best AIC   : 13.75
MAE : 0.6723
RMSE: 0.784
Corr: 0.8572

Fold 2
Best Order : (2, 0, 2)
Best AIC   : 28.81
MAE : 0.272
RMSE: 0.3557
Corr: 0.9155

Fold 3
Best Order : (2, 0, 2)
Best AIC   : 47.24
MAE : 0.2689
RMSE: 0.3581
Corr: 0.956

Fold 4
Best Order : (2, 0, 2)
Best AIC   : -3.76
MAE : 0.2323
RMSE: 0.2966
Corr: 0.9387

Fold 5
Best Order : (3, 0, 2)
Best AIC   : -65.07
MAE : 0.2435
RMSE: 0.32
Corr: 0.9665

Average
MAE : 0.3378
RMSE: 0.4229
Corr: 0.9268


In [13]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np

targets = [

    "future_nino34_lead1",
    "future_nino34_lead2",
    "future_nino34_lead3",
    "future_nino34_lead4",
    "future_nino34_lead5",
    "future_nino34_lead6"

]

for target in targets:

    print("\n" + "="*70)
    print(target)
    print("="*70)

    y = features[target]

    exog = features[
        [
            "nino3",
            "nino4",
            "iod",
            "soi"
        ]
    ]

    # ----------------------------
    # ADF Test
    # ----------------------------

    adf_result = adfuller(y.dropna())

    if adf_result[1] < 0.05:

        d = 0

    else:

        d = 1

    print("ADF p-value :", round(adf_result[1],6))
    print("Using d =", d)

    # ----------------------------
    # Candidate Orders
    # ----------------------------

    orders = [

        (1,d,0),
        (1,d,1),
        (2,d,1),
        (2,d,2),
        (3,d,1),
        (3,d,2)

    ]

    # ----------------------------
    # Cross Validation
    # ----------------------------

    tscv = TimeSeriesSplit(n_splits=5)

    mae_scores = []
    rmse_scores = []
    corr_scores = []

    for fold,(train_idx,test_idx) in enumerate(
        tscv.split(features),
        start=1
    ):

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        exog_train = exog.iloc[train_idx]
        exog_test = exog.iloc[test_idx]

        best_model = None
        best_order = None
        best_aic = np.inf

        # ----------------------------
        # Model Search
        # ----------------------------

        for order in orders:

            try:

                model = SARIMAX(

                    endog=y_train,

                    exog=exog_train,

                    order=order,

                    enforce_stationarity=False,

                    enforce_invertibility=False

                )

                fitted = model.fit(

                    disp=False,

                    maxiter=300

                )

                if fitted.aic < best_aic:

                    best_aic = fitted.aic
                    best_model = fitted
                    best_order = order

            except:

                continue

        predictions = best_model.predict(

            start=y_test.index[0],

            end=y_test.index[-1],

            exog=exog_test

        )

        mae = mean_absolute_error(
            y_test,
            predictions
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                predictions
            )
        )

        corr = np.corrcoef(
            y_test,
            predictions
        )[0,1]

        mae_scores.append(mae)
        rmse_scores.append(rmse)
        corr_scores.append(corr)

        print()

        print("Fold",fold)

        print("Best Order :",best_order)

        print("MAE :",round(mae,4))

        print("RMSE:",round(rmse,4))

        print("Corr:",round(corr,4))

    print()

    print("Average")

    print("MAE :",round(np.mean(mae_scores),4))

    print("RMSE:",round(np.mean(rmse_scores),4))

    print("Corr:",round(np.mean(corr_scores),4))


future_nino34_lead1
ADF p-value : 0.0
Using d = 0

Fold 1
Best Order : (1, 0, 0)
MAE : 0.6723
RMSE: 0.784
Corr: 0.8572

Fold 2
Best Order : (2, 0, 2)
MAE : 0.272
RMSE: 0.3557
Corr: 0.9155

Fold 3
Best Order : (2, 0, 2)
MAE : 0.2689
RMSE: 0.3581
Corr: 0.956

Fold 4
Best Order : (2, 0, 2)
MAE : 0.2323
RMSE: 0.2966
Corr: 0.9387

Fold 5
Best Order : (3, 0, 2)
MAE : 0.2435
RMSE: 0.32
Corr: 0.9665

Average
MAE : 0.3378
RMSE: 0.4229
Corr: 0.9268

future_nino34_lead2
ADF p-value : 0.0
Using d = 0

Fold 1
Best Order : (3, 0, 1)
MAE : 0.5582
RMSE: 0.6632
Corr: 0.855

Fold 2
Best Order : (2, 0, 1)
MAE : 0.4912
RMSE: 0.6332
Corr: 0.813

Fold 3
Best Order : (3, 0, 2)
MAE : 0.6235
RMSE: 0.8281
Corr: 0.5187

Fold 4
Best Order : (3, 0, 2)
MAE : 0.548
RMSE: 0.6837
Corr: 0.6486

Fold 5
Best Order : (3, 0, 2)
MAE : 0.6505
RMSE: 0.8463
Corr: 0.6571

Average
MAE : 0.5743
RMSE: 0.7309
Corr: 0.6985

future_nino34_lead3
ADF p-value : 0.0
Using d = 0

Fold 1
Best Order : (3, 0, 2)
MAE : 0.811
RMSE: 0.9511
Cor